# Brownfield to Data Product Architecture Exercise

## Scenario

Your company currently has a legacy monolithic retail database. Multiple departments use the same database directly for reporting, operations, and analytics. Over time, this has created several problems:

- Tables are tightly coupled.
- Business logic is duplicated across reports.
- Teams are unsure who owns which data.
- Changes to one table can break multiple downstream users.
- Analytics queries compete with operational workloads.
- There is no clear serving layer or data product contract.

Your goal is to assess the brownfield system, identify domain boundaries, decide what to migrate first, and design a target-state architecture where each domain owns its own data product.

```mermaid

flowchart LR

    subgraph USERS["Current Consumers"]
        APP[Operational Application]
        BI[BI Dashboards]
        ANALYSTS[Analyst SQL Queries]
        REPORTS[Scheduled Reports]
    end

    subgraph BROWNFIELD["Brownfield Monolithic System"]
        DB[(Shared Monolithic PostgreSQL Database)]

        ORDERS[(orders)]
        ORDER_ITEMS[(order_items)]
        CUSTOMERS[(customers)]
        INVENTORY[(inventory)]
        PAYMENTS[(payment_types)]
        DC[(distribution_center)]
    end

    APP --> DB
    BI --> DB
    ANALYSTS --> DB
    REPORTS --> DB

    DB --> ORDERS
    DB --> ORDER_ITEMS
    DB --> CUSTOMERS
    DB --> INVENTORY
    DB --> PAYMENTS
    DB --> DC

    ORDERS -. "No clear domain owner" .- ORDER_ITEMS
    CUSTOMERS -. "Shared directly by many consumers" .- ORDERS
    INVENTORY -. "Tightly coupled to reporting queries" .- ORDER_ITEMS
    PAYMENTS -. "Mixed operational and analytical use" .- ORDERS
    DC -. "Used through direct joins" .- INVENTORY

## Existing Brownfield Tables

The legacy monolithic retail database currently contains the following tables:

### Sales Domain Tables

| Table Name | Description |
|------------|-------------|
| orders | Order header records containing customer purchases |
| order_items | Line-item details for products included in each order |

### Customer Domain Tables

| Table Name | Description |
|------------|-------------|
| customers | Customer profile and account information |

### Inventory Domain Tables

| Table Name | Description |
|------------|-------------|
| inventory | Product inventory and stock information |

### Payments Domain Tables

| Table Name | Description |
|------------|-------------|
| payment_types | Accepted payment methods and payment classifications |

### Fulfillment Domain Tables

| Table Name | Description |
|------------|-------------|
| distribution_center | Distribution centers responsible for storing and shipping products |

---

## Existing Brownfield Relationships

| Source Table | Relationship | Target Table |
|-------------|-------------|-------------|
| orders | contains | order_items |
| order_items | references | inventory |
| orders | references | customers |
| orders | uses | payment_types |
| inventory | stored at | distribution_center |

---

## Brownfield System Observation

While these tables represent distinct business capabilities, they currently reside in a single shared PostgreSQL database. There is no clear ownership model, no formal data contracts, and consumers directly access the underlying tables. As a result, changes to one area of the database can unintentionally impact multiple applications, reports, and analytical workloads.

 ## Step 1: Assess the Current Brownfield System

Provide description of currrent state characteristics and current state anti-patterns

## Current State Characteristics

The current system has:

- One shared PostgreSQL database
- A large set of mixed-purpose tables
- Direct access by analysts, applications, and reporting tools
- No clear ownership by business domain
- No formal data contracts
- No stable serving layer

---

## Current-State Anti-Patterns

Common anti-patterns include:

- Shared database ownership
- Point-to-point reporting queries
- Direct table access by many consumers
- No clear API or view boundary
- Mixed OLTP and OLAP workloads
- Hidden business logic inside SQL reports

## Step 2: Identify Domain Boundaries

Review the existing tables and group them by business capability.

The goal is to move from table ownership to domain ownership.

## Suggested Domain Boundaries

| Domain | Example Tables | Data Product |
|----------|-------------------------|---------------------------|
| Sales Orders | orders, order_items | Sales Orders Data Product |
| Customer | customers, loyalty_membership | Customer Data Product |
| Inventory | inventory, product_catalog | Inventory Data Product |
| Distribution | distribution_center, shipments | Fulfillment Data Product |
| Payments | payment_types, transactions | Payments Data Product |

---

### Domain Ownership Goal

The objective of the migration is to move from a shared-database ownership model to a domain-oriented ownership model where each business domain owns:

- Its source tables
- Its business logic
- Its data quality rules
- Its serving layer (views, APIs, or published datasets)
- Its data contracts
- Its downstream consumer interfaces

This approach reduces coupling between teams and allows each domain to evolve independently while maintaining stable interfaces for consumers.

## Step 3: Choose What to Migrate First

A good first migration candidate should be:

- High business value
- Low-to-medium complexity
- Frequently used by consumers
- Relatively stable
- Easy to validate against the current system



### Recommended First Migration

Start with the **Sales Orders** domain.

#### Why Sales Orders?

- It directly supports revenue reporting and business performance metrics.
- It has clear business meaning and ownership.
- It contains a relatively small number of core tables (`orders` and `order_items`).
- It has well-defined relationships with other domains.
- Results can be easily validated against existing reports and dashboards.
- It provides immediate value to business users while minimizing migration risk.

### Migration Objective

The goal of the first migration is to establish the patterns that will be used for future domain migrations, including:

- Domain ownership
- Data contracts
- Serving layers
- Consumer access patterns
- Data quality controls
- Cross-domain integration standards

## Step 4: Migration Sequence

Provide a migration sequence plan

## Migration Sequence

### Phase 1: Stabilize the Existing System

The first step is to understand and document the current environment before making any architectural changes.

#### Activities

- Inventory current tables and consumers
- Identify critical reports and dashboards
- Document source tables and business owners
- Identify key dependencies and downstream consumers
- Create read-only views to reduce direct table access

#### Objective

Establish visibility into the existing system and reduce the risk of breaking business processes during migration.

---

### Phase 2: Create the First Serving Layer

The next step is to create stable interfaces between the data and its consumers.

#### Activities

- Create views for common Sales Orders reporting queries
- Define a formal Sales Orders data contract
- Document ownership, schema, and business rules
- Redirect consumers from raw tables to serving views

#### Objective

Provide a stable consumer interface that can remain unchanged while the underlying implementation evolves.

---

### Phase 3: Extract the First Domain Data Product

The Sales Orders domain becomes the first independently owned data product.

#### Activities

- Build the Sales Orders data product
- Establish ownership of tables, views, and business logic
- Implement data quality rules and validation checks
- Publish and manage the domain contract
- Validate outputs against the legacy system

#### Objective

Demonstrate the end-to-end data product pattern before expanding to additional domains.

---

### Phase 4: Add Cross-Domain Interfaces

Once the Sales Orders data product is stable, establish controlled integration points with other domains.

#### Activities

- Connect Sales Orders to Inventory using shared keys
- Connect Sales Orders to Customer using `customer_id`
- Create published serving views for cross-domain reporting
- Document cross-domain dependencies and contracts

#### Objective

Enable cross-domain analytics while maintaining clear ownership boundaries.

---

### Phase 5: Decompose Remaining Domains

After validating the migration approach, continue decomposing the monolithic database into independently owned data products.

#### Activities

- Migrate Inventory domain
- Migrate Customer domain
- Migrate Payments domain
- Migrate Distribution/Fulfillment domain
- Retire legacy dependencies as consumers transition to the new architecture

#### Objective

Complete the transition from a shared monolithic database to a domain-oriented data product architecture.

## Step 5: Migration-Aware Architecture Diagram

```mermaid
flowchart LR

    subgraph CURRENT["Current Brownfield State"]
        LEGACY_DB[Legacy Monolithic PostgreSQL Database]

        LEGACY_ORDERS[(orders)]
        LEGACY_ITEMS[(order_items)]
        LEGACY_CUSTOMERS[(customers)]
        LEGACY_INVENTORY[(inventory)]
        LEGACY_PAYMENTS[(payment_types)]
        LEGACY_DC[(distribution_center)]

        REPORTS[BI Reports / Dashboards]
        APPS[Operational Applications]
        ANALYSTS[Analyst SQL Queries]

        LEGACY_DB --> LEGACY_ORDERS
        LEGACY_DB --> LEGACY_ITEMS
        LEGACY_DB --> LEGACY_CUSTOMERS
        LEGACY_DB --> LEGACY_INVENTORY
        LEGACY_DB --> LEGACY_PAYMENTS
        LEGACY_DB --> LEGACY_DC

        REPORTS --> LEGACY_DB
        APPS --> LEGACY_DB
        ANALYSTS --> LEGACY_DB
    end

    subgraph MIGRATION["Migration Layer"]
        ASSESS[1. Assess Existing Tables and Consumers]
        BOUNDARIES[2. Identify Domain Boundaries]
        SERVING[3. Create Stable Serving Views]
        CONTRACTS[4. Define Data Contracts]
        VALIDATE[5. Validate Against Legacy Outputs]
    end

    subgraph TARGET["Target-State Data Product Architecture"]
        SALES_DP[Sales Orders Data Product]
        CUSTOMER_DP[Customer Data Product]
        INVENTORY_DP[Inventory Data Product]
        PAYMENTS_DP[Payments Data Product]
        FULFILLMENT_DP[Fulfillment Data Product]

        SALES_VIEW[Sales Serving Views]
        CUSTOMER_VIEW[Customer Serving Views]
        INVENTORY_VIEW[Inventory Serving Views]
        CROSS_VIEW[Cross-Domain Revenue View]

        CONSUMERS[Consumers: BI, Apps, Analysts]

        SALES_DP --> SALES_VIEW
        CUSTOMER_DP --> CUSTOMER_VIEW
        INVENTORY_DP --> INVENTORY_VIEW

        SALES_VIEW --> CROSS_VIEW
        CUSTOMER_VIEW --> CROSS_VIEW
        INVENTORY_VIEW --> CROSS_VIEW

        CROSS_VIEW --> CONSUMERS
        PAYMENTS_DP --> CONSUMERS
        FULFILLMENT_DP --> CONSUMERS
    end

    LEGACY_DB --> ASSESS
    ASSESS --> BOUNDARIES
    BOUNDARIES --> SERVING
    SERVING --> CONTRACTS
    CONTRACTS --> VALIDATE
    VALIDATE --> SALES_DP

    LEGACY_ORDERS -. "Migrate first" .-> SALES_DP
    LEGACY_ITEMS -. "Migrate first" .-> SALES_DP

    LEGACY_CUSTOMERS -. "Migrate second" .-> CUSTOMER_DP
    LEGACY_INVENTORY -. "Migrate third" .-> INVENTORY_DP
    LEGACY_PAYMENTS -. "Migrate later" .-> PAYMENTS_DP
    LEGACY_DC -. "Migrate later" .-> FULFILLMENT_DP